# Step 3.1 — Feature Engineering Setup

Feature engineering creates additional variables from existing features.

The goal is to represent hidden patterns in the data.

For credit risk:

A single variable may not capture the full risk.

Example:

MonthlyIncome alone does not show financial burden.

A better representation:

Debt relative to income.

Feature engineering categories:

- Ratios
- Interactions
- Aggregations
- Risk flags
- Transformations
- Categories

Each created feature must be evaluated before keeping it.

In [ ]:
# Purpose:
# Load clean processed dataset and inspect
# current feature space before engineering.

from pathlib import Path
import pandas as pd


DATA_PATH = Path(
    "../data/processed/credit_risk_clean_v1.csv"
)


df_fe = pd.read_csv(
    DATA_PATH
)


print("Shape:")
print(df_fe.shape)


print("\nColumns:")
for col in df_fe.columns:
    print(col)

# Step 3.1 — Feature Engineering Setup

Feature engineering creates new variables from existing data to capture hidden patterns.

In credit risk, individual variables often do not fully represent risk.

Examples:

- Income alone does not represent financial capacity.
- Late payment counts separately may not represent overall payment behavior.
- Debt ratio alone may not capture debt burden relative to family size.

Feature engineering categories:

1. Ratios
   - Combine related numerical variables.

2. Aggregations
   - Combine multiple related features.

3. Flags
   - Convert risk events into binary indicators.

4. Interactions
   - Capture relationships between variables.

5. Transformations
   - Improve representation of skewed variables.

6. Risk categories
   - Convert continuous risk measures into meaningful groups.

Every engineered feature will be evaluated before being kept.

In [ ]:
# Purpose:
# Load the final processed dataset and inspect
# the current feature space before engineering.

import pandas as pd


DATA_PATH = Path(
    "../data/processed/credit_risk_clean_v1.csv"
)


df_fe = pd.read_csv(
    DATA_PATH
)


print("Dataset Shape:")
print(df_fe.shape)


print("\nFeatures:")
for col in df_fe.columns:
    print("-", col)

# Feature Engineering — Income Per Dependent

## Feature Idea

MonthlyIncome represents earning capacity, but it does not consider household size.

A person with the same income but more dependents may have a higher financial burden.

New feature:

IncomePerDependent

Formula:

IncomePerDependent =
MonthlyIncome / (NumberOfDependents + 1)

The +1 avoids division by zero for customers with no dependents.

Evaluation:

The feature will be compared against the target variable:

SeriousDlqin2yrs

If it provides useful separation between default and non-default customers, it will be considered for the final feature set.

In [ ]:
# Purpose:
# Create IncomePerDependent feature and
# analyze its relationship with default risk.

df_fe["IncomePerDependent"] = (
    df_fe["MonthlyIncome"] /
    (df_fe["NumberOfDependents"] + 1)
)


print(df_fe["IncomePerDependent"].describe())


print("\nDefault rate by IncomePerDependent quartile:")

df_fe["IncomePerDependent_bin"] = pd.qcut(
    df_fe["IncomePerDependent"],
    q=5,
    duplicates="drop"
)


income_dependents_analysis = (
    df_fe.groupby(
        "IncomePerDependent_bin",
        observed=True
    )["SeriousDlqin2yrs"]
    .mean()
)


income_dependents_analysis

# IncomePerDependent Feature Decision

Feature created:

`IncomePerDependent`

Purpose:

Measure available income relative to household dependency burden.

Evidence:

Default rate decreases consistently as IncomePerDependent increases.

Findings:

- Lowest income-per-dependent group:
  Default rate = 10.26%

- Highest income-per-dependent group:
  Default rate = 4.34%

Decision:

KEEP

Reason:

IncomePerDependent provides additional financial context beyond MonthlyIncome and shows a meaningful relationship with credit default risk.

# Total Late Payments Feature

## Feature Idea

The dataset contains three separate delinquency variables:

- NumberOfTime30-59DaysPastDueNotWorse
- NumberOfTime60-89DaysPastDueNotWorse
- NumberOfTimes90DaysLate

Each feature represents a different severity level of late payment.

However, overall payment behavior may be better represented by combining all delinquency events.

New feature:

TotalLatePayments

Formula:

TotalLatePayments =
30-59 days late
+
60-89 days late
+
90+ days late

Purpose:

Create a single measure of a borrower's historical payment problems.

Evaluation:

The feature will be analyzed by comparing default rates across risk groups.

In [ ]:
# Purpose:
# Create TotalLatePayments feature and
# analyze its relationship with default risk.


df_fe["TotalLatePayments"] = (
    df_fe["NumberOfTime30-59DaysPastDueNotWorse"]
    +
    df_fe["NumberOfTime60-89DaysPastDueNotWorse"]
    +
    df_fe["NumberOfTimes90DaysLate"]
)


print(
    df_fe["TotalLatePayments"].describe()
)


print("\nDefault rate by TotalLatePayments groups:")


df_fe["TotalLatePayments_bin"] = pd.qcut(
    df_fe["TotalLatePayments"],
    q=5,
    duplicates="drop"
)


total_late_analysis = (
    df_fe.groupby(
        "TotalLatePayments_bin",
        observed=True
    )["SeriousDlqin2yrs"]
    .mean()
)


total_late_analysis

# TotalLatePayments Feature Decision

Feature created:

`TotalLatePayments`

Purpose:

Combine all delinquency events into a single measure of payment history risk.

Formula:

TotalLatePayments =
30-59 days late
+
60-89 days late
+
90+ days late

Evidence:

Default rates:

- Low delinquency group:
  4.02%

- Higher delinquency group:
  35.09%

The feature shows a strong relationship with credit default risk.

Decision:

KEEP

Reason:

TotalLatePayments provides a compact representation of repayment behavior and captures cumulative delinquency risk.

# AnyLatePayment Feature

## Feature Idea

TotalLatePayments measures the number of delinquency events.

However, in credit risk, the existence of any previous late payment can itself be an important risk signal.

A customer with:

- 0 late payments
- 1 or more late payments

may represent two very different risk groups.

New feature:

`AnyLatePayment`

Formula:

AnyLatePayment =
1 if TotalLatePayments > 0

AnyLatePayment =
0 if TotalLatePayments = 0

Purpose:

Capture whether the borrower has ever shown delinquency behavior.

Evaluation:

The feature will be compared against the default rate.

In [ ]:
# Purpose:
# Create AnyLatePayment binary risk flag and
# analyze its relationship with default risk.


df_fe["AnyLatePayment"] = (
    df_fe["TotalLatePayments"] > 0
).astype(int)


print(
    df_fe["AnyLatePayment"]
    .value_counts()
)


print("\nDefault rate by AnyLatePayment:")


any_late_analysis = (
    df_fe.groupby(
        "AnyLatePayment"
    )["SeriousDlqin2yrs"]
    .mean()
)


any_late_analysis

# AnyLatePayment Feature Decision

Feature created:

`AnyLatePayment`

Purpose:

Capture whether a customer has any history of payment delinquency.

Formula:

AnyLatePayment =
1 if TotalLatePayments > 0

0 otherwise

Evidence:

Default rates:

- No late payment history:
  2.84%

- Has late payment history:
  21.98%

Customers with late payment history showed approximately 7.7x higher default risk.

Decision:

KEEP

Reason:

The feature provides a strong behavioral risk indicator and improves interpretability of delinquency history.

# SevereDelinquencyFlag Feature

## Feature Idea

Not all late payments have the same risk level.

The dataset contains:

- 30-59 days late
- 60-89 days late
- 90+ days late

A 90+ day delinquency represents severe financial distress.

New feature:

`SevereDelinquencyFlag`

Formula:

SevereDelinquencyFlag =
1 if NumberOfTimes90DaysLate > 0

0 otherwise

Purpose:

Identify customers who have experienced severe payment problems.

Evaluation:

The feature will be analyzed by comparing default rates between customers with and without severe delinquency history.

In [ ]:
# Purpose:
# Create SevereDelinquencyFlag binary feature and
# analyze its relationship with default risk.


df_fe["SevereDelinquencyFlag"] = (
    df_fe["NumberOfTimes90DaysLate"] > 0
).astype(int)


print(
    df_fe["SevereDelinquencyFlag"]
    .value_counts()
)


print("\nDefault rate by SevereDelinquencyFlag:")


severe_delinq_analysis = (
    df_fe.groupby(
        "SevereDelinquencyFlag"
    )["SeriousDlqin2yrs"]
    .mean()
)


severe_delinq_analysis

# HighDebtRatioFlag Feature

## Feature Idea

DebtRatio represents the borrower's debt burden.

However, a continuous ratio may not always be easy for a model to interpret.

A binary risk flag can capture whether a customer belongs to a higher debt burden group.

New feature:

`HighDebtRatioFlag`

Formula:

HighDebtRatioFlag =
1 if DebtRatio > median DebtRatio

0 otherwise

Purpose:

Identify customers with relatively higher debt burden compared with the rest of the population.

Evaluation:

The feature will be analyzed by comparing default rates between low and high debt ratio groups.

In [ ]:
# Purpose:
# Create HighDebtRatioFlag feature using the
# dataset median DebtRatio as the threshold.

debt_ratio_threshold = (
    df_fe["DebtRatio"]
    .median()
)


df_fe["HighDebtRatioFlag"] = (
    df_fe["DebtRatio"] >
    debt_ratio_threshold
).astype(int)


print("DebtRatio threshold:")
print(debt_ratio_threshold)


print("\nHighDebtRatioFlag distribution:")
print(
    df_fe["HighDebtRatioFlag"]
    .value_counts()
)


print("\nDefault rate by HighDebtRatioFlag:")


high_debt_analysis = (
    df_fe.groupby(
        "HighDebtRatioFlag"
    )["SeriousDlqin2yrs"]
    .mean()
)


high_debt_analysis

# HighDebtRatioFlag Feature Decision

Feature created:

`HighDebtRatioFlag`

Purpose:

Identify customers with relatively high debt burden.

Formula:

HighDebtRatioFlag =
1 if DebtRatio > median DebtRatio

0 otherwise


Evidence:

Default rates:

- Lower debt burden:
  5.85%

- Higher debt burden:
  7.52%


The feature shows a moderate relationship with default risk.

Decision:

KEEP TEMPORARILY

Reason:

The feature has business value, but final selection will depend on model-based feature importance because DebtRatio and DebtRatio_log already represent similar information.

# HighUtilizationFlag Feature

## Feature Idea

RevolvingUtilizationOfUnsecuredLines represents how much of the available unsecured credit a customer is using.

High credit utilization can indicate:

- Increased dependence on credit.
- Reduced financial flexibility.
- Higher repayment pressure.

Instead of only using the continuous utilization value, create a risk flag.

New feature:

`HighUtilizationFlag`

Formula:

HighUtilizationFlag =
1 if RevolvingUtilizationOfUnsecuredLines > 0.8

0 otherwise


Purpose:

Identify customers with high credit usage.

Evaluation:

The feature will be analyzed by comparing default rates between customers with normal and high utilization.

In [ ]:
# Purpose:
# Create HighUtilizationFlag feature and
# analyze its relationship with default risk.


utilization_threshold = 0.8


df_fe["HighUtilizationFlag"] = (
    df_fe["RevolvingUtilizationOfUnsecuredLines"]
    > utilization_threshold
).astype(int)


print("Utilization threshold:")
print(utilization_threshold)


print("\nHighUtilizationFlag distribution:")
print(
    df_fe["HighUtilizationFlag"]
    .value_counts()
)


print("\nDefault rate by HighUtilizationFlag:")


high_utilization_analysis = (
    df_fe.groupby(
        "HighUtilizationFlag"
    )["SeriousDlqin2yrs"]
    .mean()
)


high_utilization_analysis

# HighUtilizationFlag Feature Decision

Feature created:

`HighUtilizationFlag`

Purpose:

Identify customers with high unsecured credit utilization.

Formula:

HighUtilizationFlag =
1 if RevolvingUtilizationOfUnsecuredLines > 0.8

0 otherwise


Evidence:

Default rates:

- Normal utilization:
  3.79%

- High utilization:
  21.22%


Customers with high utilization showed approximately 5.6x higher default risk.

Decision:

KEEP

Reason:

The feature captures credit stress behavior and provides a strong interpretable risk indicator.

# CreditExposure Feature

## Feature Idea

The dataset contains two variables related to existing credit obligations:

- NumberOfOpenCreditLinesAndLoans
- NumberRealEstateLoansOrLines

Individually, they represent different types of credit exposure.

A borrower with many active credit accounts may have greater financial commitments.

New feature:

`CreditExposure`

Formula:

CreditExposure =
NumberOfOpenCreditLinesAndLoans
+
NumberRealEstateLoansOrLines


Purpose:

Create a combined measure of total credit accounts.

Evaluation:

The feature will be analyzed by comparing default rates across different credit exposure levels.

In [ ]:
# Purpose:
# Create CreditExposure feature and
# analyze its relationship with default risk.


df_fe["CreditExposure"] = (
    df_fe["NumberOfOpenCreditLinesAndLoans"]
    +
    df_fe["NumberRealEstateLoansOrLines"]
)


print(
    df_fe["CreditExposure"].describe()
)


print("\nDefault rate by CreditExposure groups:")


df_fe["CreditExposure_bin"] = pd.qcut(
    df_fe["CreditExposure"],
    q=5,
    duplicates="drop"
)


credit_exposure_analysis = (
    df_fe.groupby(
        "CreditExposure_bin",
        observed=True
    )["SeriousDlqin2yrs"]
    .mean()
)


credit_exposure_analysis

# CreditExposure Feature Decision

Feature created:

`CreditExposure`

Purpose:

Combine different types of credit obligations into one measure.

Formula:

CreditExposure =
NumberOfOpenCreditLinesAndLoans
+
NumberRealEstateLoansOrLines


Evidence:

Default rate pattern:

- Lowest exposure group:
  9.01%

- Middle exposure group:
  5.19%

- Highest exposure group:
  7.04%


The relationship is non-linear.

Low exposure may indicate limited credit history, while very high exposure may indicate excessive borrowing.

Decision:

KEEP TEMPORARILY

Reason:

The feature provides additional credit exposure information, but final selection will be performed using model-based feature importance methods.

# HasDependents Feature

## Feature Idea

NumberOfDependents is a count feature.

However, the presence of dependents itself may represent additional financial responsibility.

A customer with dependents may have:

- Higher household expenses.
- Less disposable income.
- Greater financial obligations.

New feature:

`HasDependents`

Formula:

HasDependents =
1 if NumberOfDependents > 0

0 otherwise


Purpose:

Capture whether the borrower has dependent-related financial responsibility.

Evaluation:

The feature will be analyzed by comparing default rates between customers with and without dependents.

In [ ]:
# Purpose:
# Create HasDependents feature and
# analyze its relationship with default risk.


df_fe["HasDependents"] = (
    df_fe["NumberOfDependents"] > 0
).astype(int)


print(
    df_fe["HasDependents"]
    .value_counts()
)


print("\nDefault rate by HasDependents:")


has_dependents_analysis = (
    df_fe.groupby(
        "HasDependents"
    )["SeriousDlqin2yrs"]
    .mean()
)


has_dependents_analysis

# HasDependents Feature Decision

Feature created:

`HasDependents`

Purpose:

Identify whether a borrower has dependent-related financial responsibility.

Formula:

HasDependents =
1 if NumberOfDependents > 0

0 otherwise


Evidence:

Default rates:

- No dependents:
  5.81%

- Has dependents:
  8.03%


Customers with dependents show higher default risk.

Decision:

KEEP TEMPORARILY

Reason:

The feature provides a simple household responsibility indicator, but it may overlap with NumberOfDependents and IncomePerDependent.

# Step 3.2 — Domain Features

# DelinquencySeverityScore Feature

## Feature Idea

The previous features count delinquency events, but they treat all late payments equally.

However, credit risk severity is different:

- 30-59 days late indicates early payment problems.
- 60-89 days late indicates more serious problems.
- 90+ days late indicates severe financial distress.

A weighted score can represent the seriousness of delinquency history.

New feature:

`DelinquencySeverityScore`

Formula:

DelinquencySeverityScore =

(NumberOfTime30-59DaysPastDueNotWorse × 1)
+
(NumberOfTime60-89DaysPastDueNotWorse × 2)
+
(NumberOfTimes90DaysLate × 3)


Purpose:

Create a severity-based measure of repayment problems.

Evaluation:

The feature will be analyzed by comparing default rates across severity groups.

In [ ]:
# Purpose:
# Create DelinquencySeverityScore feature and
# analyze its relationship with default risk.


df_fe["DelinquencySeverityScore"] = (
    df_fe["NumberOfTime30-59DaysPastDueNotWorse"] * 1
    +
    df_fe["NumberOfTime60-89DaysPastDueNotWorse"] * 2
    +
    df_fe["NumberOfTimes90DaysLate"] * 3
)


print(
    df_fe["DelinquencySeverityScore"].describe()
)


print("\nDefault rate by DelinquencySeverityScore groups:")


df_fe["DelinquencySeverityScore_bin"] = pd.qcut(
    df_fe["DelinquencySeverityScore"],
    q=5,
    duplicates="drop"
)


severity_analysis = (
    df_fe.groupby(
        "DelinquencySeverityScore_bin",
        observed=True
    )["SeriousDlqin2yrs"]
    .mean()
)


severity_analysis

# DelinquencySeverityScore Feature Decision

Feature created:

`DelinquencySeverityScore`

Purpose:

Represent the seriousness of delinquency history.

Formula:

30-59 days late × 1
+
60-89 days late × 2
+
90+ days late × 3


Evidence:

Default rates:

- Low severity group:
  3.46%

- Higher severity group:
  31.39%


The feature shows strong separation between low-risk and high-risk customers.

Decision:

KEEP TEMPORARILY

Reason:

The feature captures delinquency severity, but final feature selection will determine whether it provides additional information beyond existing delinquency features.

# AgeRiskCategory Feature

## Feature Idea

Age is currently represented as a continuous variable.

However, credit behavior may not change linearly with age.

Different age groups may have different financial patterns:

- Younger borrowers may have less credit history.
- Middle-aged borrowers may have more stable income.
- Older borrowers may have different financial obligations.

New feature:

`AgeRiskCategory`

Categories:

0 = Young (<30)

1 = Middle (30-60)

2 = Senior (>60)


Purpose:

Capture possible nonlinear relationship between age groups and default risk.

Evaluation:

Default rates will be compared across age categories.

In [ ]:
# Purpose:
# Create AgeRiskCategory feature and
# analyze default risk across age groups.


def create_age_category(age):
    if age < 30:
        return 0
    elif age <= 60:
        return 1
    else:
        return 2


df_fe["AgeRiskCategory"] = (
    df_fe["age"]
    .apply(create_age_category)
)


print(
    df_fe["AgeRiskCategory"]
    .value_counts()
)


print("\nDefault rate by AgeRiskCategory:")


age_category_analysis = (
    df_fe.groupby(
        "AgeRiskCategory"
    )["SeriousDlqin2yrs"]
    .mean()
)


age_category_analysis

# AgeRiskCategory Feature Decision

Feature created:

`AgeRiskCategory`

Categories:

0 = Young (<30)

1 = Middle (30-60)

2 = Senior (>60)


Purpose:

Capture nonlinear age-related default behavior.


Evidence:

Default rates:

- Young:
  11.73%

- Middle:
  7.95%

- Senior:
  2.99%


Age groups show different risk profiles.

Decision:

KEEP TEMPORARILY

Reason:

The feature captures nonlinear age patterns and may improve models that cannot automatically learn age thresholds.

# DebtIncomeInteraction Feature

## Feature Idea

Debt burden should not be interpreted independently from income.

A high DebtRatio may have different meanings:

Example:

Customer A:
- High debt
- High income

Customer B:
- High debt
- Low income

Customer B is likely under more financial pressure.

New feature:

`DebtIncomeInteraction`

Formula:

DebtIncomeInteraction =

DebtRatio × MonthlyIncome_log


Purpose:

Capture combined effect of debt burden and income capacity.

Evaluation:

The feature will be analyzed by comparing default rates across interaction levels.

In [ ]:
# Purpose:
# Create DebtIncomeInteraction feature and
# analyze its relationship with default risk.


df_fe["DebtIncomeInteraction"] = (
    df_fe["DebtRatio_log"]
    *
    df_fe["MonthlyIncome_log"]
)


print(
    df_fe["DebtIncomeInteraction"]
    .describe()
)


print("\nDefault rate by DebtIncomeInteraction groups:")


df_fe["DebtIncomeInteraction_bin"] = pd.qcut(
    df_fe["DebtIncomeInteraction"],
    q=5,
    duplicates="drop"
)


debt_income_analysis = (
    df_fe.groupby(
        "DebtIncomeInteraction_bin",
        observed=True
    )["SeriousDlqin2yrs"]
    .mean()
)


debt_income_analysis

# DebtIncomeInteraction Feature Decision

Feature created:

`DebtIncomeInteraction`

Formula:

DebtRatio_log × MonthlyIncome_log


Purpose:

Capture combined effect of debt burden and income capacity.


Evidence:

Default rates:

- Lowest group:
  6.04%

- Highest risk group:
  9.27%

The feature shows some separation, but the relationship is not consistently increasing across groups.


Decision:

KEEP TEMPORARILY

Reason:

The feature has domain value but may overlap with existing debt and income features.
Final selection will be performed using model-based feature importance.

In [ ]:
# Load original raw dataset temporarily

raw_df = pd.read_csv(
    "../data/raw/cs-training.csv"
)


raw_df.shape

In [ ]:
# Create missingness indicator before imputation

raw_df["IncomeMissingFlag"] = (
    raw_df["MonthlyIncome"]
    .isna()
    .astype(int)
)


raw_df["IncomeMissingFlag"].value_counts()

In [ ]:
raw_df.groupby(
    "IncomeMissingFlag"
)["SeriousDlqin2yrs"].mean()

In [ ]:
df_fe = df_fe.merge(
    raw_df[
        [
            "Unnamed: 0",
            "IncomeMissingFlag"
        ]
    ],
    on="Unnamed: 0",
    how="left"
)

In [ ]:
df_fe["IncomeMissingFlag"].value_counts()

df_fe[
    [
        "MonthlyIncome",
        "IncomeMissingFlag"
    ]
].head()

In [ ]:
df_fe.shape

In [ ]:
# Save final feature engineering dataset

output_path = "../data/interim/feature_engineering_v1.csv"

df_fe.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", df_fe.shape)

In [ ]:
# Check final columns

df_fe.columns.tolist()

In [ ]:
df_fe.isna().sum()

# feature selection

In [ ]:
# Correlation analysis

import matplotlib.pyplot as plt
import seaborn as sns


# Select only numeric features for correlation

numeric_df = df_fe.select_dtypes(
    include=["int64", "float64"]
)


corr_matrix = numeric_df.corr()


corr_matrix.shape

In [ ]:
plt.figure(figsize=(14,10))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0
)

plt.title("Feature Correlation Matrix")

plt.show()

In [ ]:
corr_pairs = (
    corr_matrix
    .abs()
    .unstack()
    .sort_values(
        ascending=False
    )
)


corr_pairs[
    corr_pairs < 1
].head(27)

# Step 3.3.1 — Correlation Analysis

## Purpose

Correlation analysis is used to identify:

- Highly correlated features.
- Duplicate information.
- Potential multicollinearity problems.
- Features that may provide overlapping signals.

Important:

High correlation does not automatically mean a feature should be removed.

Decision depends on:

- Model type.
- Predictive performance.
- Feature importance.
- Explainability requirements.

---

# Correlation Findings

## 1. CreditExposure vs NumberOfOpenCreditLinesAndLoans

Correlation:

0.984

Reason:

CreditExposure was created from:

CreditExposure =
NumberOfOpenCreditLinesAndLoans
+
NumberRealEstateLoansOrLines


Decision:

Candidate for removal.

Reason:

It provides almost the same information as the original credit line features.

---

## 2. DebtIncomeInteraction vs DebtRatio_log

Correlation:

0.958

Reason:

The interaction feature is strongly influenced by DebtRatio_log.

Decision:

Keep temporarily.

Reason:

Final removal decision will depend on model importance methods.

---

## 3. DelinquencySeverityScore vs TotalLatePayments

Correlation:

0.939

Reason:

Both represent delinquency behavior.

Difference:

- TotalLatePayments measures frequency.
- DelinquencySeverityScore measures weighted severity.

Decision:

Keep temporarily.

---

## 4. AgeRiskCategory vs Age

Correlation:

0.843

Reason:

AgeRiskCategory is created from age ranges.

Decision:

Keep temporarily.

Reason:

Useful for models that cannot easily learn nonlinear age groups.

---

## 5. HasDependents vs NumberOfDependents

Correlation:

0.825

Reason:

HasDependents is derived from NumberOfDependents.

Decision:

Candidate for removal.

Reason:

NumberOfDependents contains more detailed information.

---

## 6. TotalLatePayments vs NumberOfTime30-59DaysPastDueNotWorse

Correlation:

0.821

Decision:

Keep temporarily.

Reason:

Although correlated, they may capture different repayment patterns.

---

## 7. IncomePerDependent vs MonthlyIncome

Correlation:

0.817

Decision:

Keep.

Reason:

They represent different concepts:

- MonthlyIncome = earning capacity.
- IncomePerDependent = financial burden.

---

## 8. HighUtilizationFlag vs RevolvingUtilization_log

Correlation:

0.773

Decision:

Keep temporarily.

Reason:

Different representation:

- HighUtilizationFlag captures threshold risk.
- RevolvingUtilization_log captures continuous behavior.

---

# Correlation-Based Feature Review Summary

| Feature | Decision |
|---|---|
| CreditExposure | Candidate removal |
| DebtIncomeInteraction | Keep temporarily |
| DelinquencySeverityScore | Keep temporarily |
| AgeRiskCategory | Keep temporarily |
| HasDependents | Candidate removal |
| IncomePerDependent | Keep |
| HighUtilizationFlag | Keep |
| TotalLatePayments | Keep |

---

# Remove Analysis Columns

The following columns were created only for analysis and should not be used as model features:

- IncomePerDependent_bin
- TotalLatePayments_bin
- CreditExposure_bin
- DebtIncomeInteraction_bin
- DelinquencySeverityScore_bin

These columns must be removed before model training.

---

# Next Step

## Step 3.3.2 — Mutual Information

Mutual Information will measure nonlinear relationships between features and the target:

Target:

`SeriousDlqin2yrs`

Unlike correlation, Mutual Information can detect complex nonlinear patterns.

# Step 3.3.2 — Mutual Information Feature Selection

## Purpose

Mutual Information measures the dependency between each feature and the target variable.

Unlike correlation:

- Correlation detects linear relationships.
- Mutual Information detects nonlinear relationships.

Target:

`SeriousDlqin2yrs`

Interpretation:

Higher Mutual Information score:

→ More information about default risk.

Lower Mutual Information score:

→ Less predictive information.

The results will be used together with:

- Correlation analysis
- Feature importance
- Permutation importance
- SHAP importance

to create the final feature set.

In [ ]:
# Mutual Information Feature Selection

from sklearn.feature_selection import mutual_info_classif


# Remove target and analysis columns

drop_cols = [
    "SeriousDlqin2yrs",
    "IncomePerDependent_bin",
    "TotalLatePayments_bin",
    "CreditExposure_bin",
    "DebtIncomeInteraction_bin",
    "DelinquencySeverityScore_bin"
]


X_mi = df_fe.drop(
    columns=drop_cols,
    errors="ignore"
)

y_mi = df_fe["SeriousDlqin2yrs"]


# Calculate Mutual Information

mi_scores = mutual_info_classif(
    X_mi,
    y_mi,
    random_state=42
)


mi_results = (
    pd.DataFrame(
        {
            "Feature": X_mi.columns,
            "Mutual_Information": mi_scores
        }
    )
    .sort_values(
        by="Mutual_Information",
        ascending=False
    )
)


mi_results

In [ ]:
plt.figure(figsize=(10,8))

sns.barplot(
    data=mi_results,
    x="Mutual_Information",
    y="Feature"
)

plt.title("Mutual Information Feature Importance")

plt.show()

# Mutual Information Feature Selection Results

## Purpose

Mutual Information was used to measure nonlinear dependency between each feature and:

`SeriousDlqin2yrs`

Higher scores indicate features containing more information about default risk.

---

# Findings

The strongest predictive features were:

1. DelinquencySeverityScore
2. TotalLatePayments
3. AnyLatePayment
4. RevolvingUtilization_log
5. RevolvingUtilizationOfUnsecuredLines
6. NumberOfTimes90DaysLate
7. SevereDelinquencyFlag
8. HighUtilizationFlag

The results confirm that:

- Payment history is the strongest risk signal.
- Credit utilization is a major predictor.

---

# Feature Decisions

Keep:

- Delinquency features
- Utilization features

Test further:

- Income features
- Debt features
- Age features

Remove:

`Unnamed: 0`

Reason:

It is only an identifier and has no predictive meaning.

---

Next:

Step 3.3.3 — Model Feature Importance

A tree-based model will learn feature importance from actual prediction performance.

# Step 3.3.3 — Model Feature Importance

## Purpose

Tree-based feature importance is used to measure how much each feature contributes to model decisions.

Tree models can capture:

- Nonlinear relationships.
- Feature interactions.
- Complex risk patterns.

Model used:

LightGBM / XGBoost classifier

Output:

A ranked list of features based on importance.

These results will be combined with:

- Correlation analysis
- Mutual Information
- Permutation Importance
- SHAP importance

to select the final feature set.

In [ ]:
# Prepare data for feature importance

from lightgbm import LGBMClassifier


# Remove target and non-feature columns

drop_cols = [
    "SeriousDlqin2yrs",
    "Unnamed: 0",
    "IncomePerDependent_bin",
    "TotalLatePayments_bin",
    "CreditExposure_bin",
    "DebtIncomeInteraction_bin",
    "DelinquencySeverityScore_bin"
]


X = df_fe.drop(
    columns=drop_cols,
    errors="ignore"
)

y = df_fe["SeriousDlqin2yrs"]


print(X.shape)

In [ ]:
feature_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    class_weight="balanced"
)


feature_model.fit(
    X,
    y
)

In [ ]:
importance_df = (
    pd.DataFrame(
        {
            "Feature": X.columns,
            "Importance": feature_model.feature_importances_
        }
    )
    .sort_values(
        by="Importance",
        ascending=False
    )
)


importance_df

In [ ]:
plt.figure(figsize=(10,8))

sns.barplot(
    data=importance_df,
    x="Importance",
    y="Feature"
)

plt.title(
    "LightGBM Feature Importance"
)

plt.show()

# Model Feature Importance Results

## Method

A LightGBM classifier was trained to measure feature contribution.

Tree-based importance captures:

- Nonlinear relationships.
- Feature interactions.
- Predictive contribution.

---

# Findings

Top predictive features:

1. RevolvingUtilizationOfUnsecuredLines
2. age
3. DebtRatio
4. IncomePerDependent
5. MonthlyIncome
6. DebtIncomeInteraction
7. NumberOfOpenCreditLinesAndLoans
8. CreditExposure
9. NumberRealEstateLoansOrLines
10. DelinquencySeverityScore


The model confirms that:

- Credit utilization.
- Debt burden.
- Income capacity.
- Delinquency history.

are major drivers of credit default risk.


Features with zero importance:

- RevolvingUtilization_log
- MonthlyIncome_log
- DebtRatio_log
- HighDebtRatioFlag

These features provide little additional value because their information is already captured by original variables.

# Step 3.3.4 — Permutation Importance

## Purpose

Permutation importance measures how much each feature affects model performance.

Method:

1. Train a model.
2. Randomly shuffle one feature.
3. Evaluate performance change.

Interpretation:

Large performance decrease:

→ Important feature.

Small or no change:

→ Feature provides little additional information.

Permutation importance is model-agnostic and will be compared with:

- Correlation analysis
- Mutual Information
- LightGBM Feature Importance
- SHAP Importance

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance


# Prepare train validation split

X_pi = df_fe.drop(
    columns=[
        "SeriousDlqin2yrs",
        "Unnamed: 0",
        "IncomePerDependent_bin",
        "TotalLatePayments_bin",
        "CreditExposure_bin",
        "DebtIncomeInteraction_bin",
        "DelinquencySeverityScore_bin"
    ],
    errors="ignore"
)

y_pi = df_fe["SeriousDlqin2yrs"]


X_train_pi, X_valid_pi, y_train_pi, y_valid_pi = train_test_split(
    X_pi,
    y_pi,
    test_size=0.2,
    stratify=y_pi,
    random_state=42
)


print("Train:", X_train_pi.shape)
print("Validation:", X_valid_pi.shape)

In [ ]:
from lightgbm import LGBMClassifier


perm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    class_weight="balanced"
)


perm_model.fit(
    X_train_pi,
    y_train_pi
)

In [ ]:
perm_result = permutation_importance(
    perm_model,
    X_valid_pi,
    y_valid_pi,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)


perm_df = (
    pd.DataFrame(
        {
            "Feature": X_valid_pi.columns,
            "Importance_Mean": perm_result.importances_mean,
            "Importance_STD": perm_result.importances_std
        }
    )
    .sort_values(
        by="Importance_Mean",
        ascending=False
    )
)


perm_df

# Step 3.3.4 — Permutation Importance Results

## Purpose

Permutation importance measures the effect of each feature on model performance.

Method:

1. Train LightGBM model.
2. Shuffle one feature at a time.
3. Measure ROC-AUC decrease.

A larger decrease means the feature is more important.

---

# Findings

The strongest features were:

1. RevolvingUtilizationOfUnsecuredLines
2. DelinquencySeverityScore
3. TotalLatePayments
4. age
5. DebtIncomeInteraction

These results confirm previous methods:

- Credit utilization is the strongest predictor.
- Payment history is highly predictive.
- Debt and income relationships provide additional information.

---

# Low Value Features

Features with almost zero permutation impact:

- HasDependents
- MonthlyIncome_log
- RevolvingUtilization_log
- DebtRatio_log
- AgeRiskCategory

These features provide little additional predictive value because their information is already represented by other variables.

---

Next:

Step 3.3.5 — Recursive Feature Elimination (RFE)

# Step 3.3.5 — Recursive Feature Elimination (RFE)

## Purpose

Recursive Feature Elimination selects the most useful subset of features.

Process:

1. Train a model using all features.
2. Rank features by importance.
3. Remove the least important features.
4. Repeat until the selected number of features is reached.

RFE helps identify the optimal feature subset for model performance.

The results will be compared with:

- Correlation Analysis
- Mutual Information
- Feature Importance
- Permutation Importance
- SHAP Importance

In [ ]:
from sklearn.feature_selection import RFE
from lightgbm import LGBMClassifier


# Prepare dataset

X_rfe = df_fe.drop(
    columns=[
        "SeriousDlqin2yrs",
        "Unnamed: 0",
        "IncomePerDependent_bin",
        "TotalLatePayments_bin",
        "CreditExposure_bin",
        "DebtIncomeInteraction_bin",
        "DelinquencySeverityScore_bin"
    ],
    errors="ignore"
)

y_rfe = df_fe["SeriousDlqin2yrs"]


print(X_rfe.shape)

In [ ]:
rfe_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    class_weight="balanced"
)

In [ ]:
feature_counts = [
    5,
    10,
    15,
    20
]


rfe_results = []


for n in feature_counts:

    rfe = RFE(
        estimator=rfe_model,
        n_features_to_select=n
    )

    rfe.fit(
        X_rfe,
        y_rfe
    )

    selected_features = X_rfe.columns[
        rfe.support_
    ]

    rfe_results.append(
        {
            "Features_Selected": n,
            "Features": list(selected_features)
        }
    )


rfe_results

In [ ]:
rfe.ranking_

In [ ]:
rfe_ranking = pd.DataFrame(
    {
        "Feature": X_rfe.columns,
        "Ranking": rfe.ranking_
    }
).sort_values(
    by="Ranking"
)


rfe_ranking

# Step 3.3.5 — Recursive Feature Elimination Results

## Purpose

RFE was used to identify the most useful subset of features by repeatedly removing weaker features based on model importance.

Model:

LightGBM Classifier


# Findings

RFE selected most original credit features and removed mainly transformed features.

Removed candidates:

- MonthlyIncome_log
- DebtRatio_log
- RevolvingUtilization_log

These features were already represented by original variables.


Features requiring further validation:

- HasDependents
- HighDebtRatioFlag
- HighUtilizationFlag
- AgeRiskCategory
- SevereDelinquencyFlag
- AnyLatePayment


Final decision will be made after SHAP importance analysis.

# Step 3.3.6 — SHAP Feature Importance

## Purpose

SHAP importance explains how each feature contributes to model predictions.

SHAP provides:

- Global feature ranking.
- Direction of influence.
- Individual prediction explanations.

Unlike traditional feature importance, SHAP shows whether a feature increases or decreases the probability of default.

The SHAP results will be combined with:

- Correlation Analysis
- Mutual Information
- Model Feature Importance
- Permutation Importance
- RFE

to create the final feature set.

In [ ]:
import shap


# Use trained permutation model

X_shap = X_valid_pi.copy()


# Create SHAP explainer

explainer = shap.TreeExplainer(
    perm_model
)


shap_values = explainer.shap_values(
    X_shap
)

In [ ]:
shap_importance = (
    pd.DataFrame(
        {
            "Feature": X_shap.columns,
            "SHAP_Importance": abs(shap_values).mean(axis=0)
        }
    )
    .sort_values(
        by="SHAP_Importance",
        ascending=False
    )
)


shap_importance

In [ ]:
shap.summary_plot(
    shap_values,
    X_shap
)

In [ ]:
final_features = [
    "RevolvingUtilizationOfUnsecuredLines",
    "TotalLatePayments",
    "DelinquencySeverityScore",
    "age",
    "NumberOfOpenCreditLinesAndLoans",
    "MonthlyIncome",
    "NumberRealEstateLoansOrLines",
    "DebtIncomeInteraction",
    "DebtRatio",
    "CreditExposure",
    "IncomePerDependent",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
    "IncomeMissingFlag",
    "SeriousDlqin2yrs"
]


df_final = df_fe[final_features]


output_path = "../data/processed/final_features_v1.csv"

df_final.to_csv(
    output_path,
    index=False
)


print("Saved:", output_path)
print("Shape:", df_final.shape)